### GAN'S-Generative Adversial Networks

### Vanilla GAN

In [1]:
# !pip install torchvision

In [2]:
import torch
import os
from torch.utils.data import DataLoader , Dataset
from torchvision import transforms
from PIL import Image

In [12]:
# Image load => transform => dataset of all images
class ImageProcessor:
    def __init__(self,root_dir_path,transformations=None):
        self.root_dir_path=root_dir_path
        self.transformations=transformations

        # list of path for all images
        self.all_img_paths=[os.path.join(root_dir_path,img) for img in os.listdir(root_dir_path)]

    def __len__(self):
        return len(self.all_img_paths)

    def __getitem__(self,idx):
        img_path=self.all_img_paths[idx]
        img=Image.open(img_path).convert("RGB")

        if self.transformations:
            img=self.transformations(img)

        return img

In [6]:
root_dir_path="./img_align_celeba"

transformations=transforms.Compose(
    [
        transforms.CenterCrop(178), # 178x218 => 178x178
        transforms.Resize(64), # 64x64
        transforms.ToTensor(),
        transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)) # [-1,1]
    ]
)

In [14]:
dataset=ImageProcessor(root_dir_path,transformations)
print(f"loaded {len(dataset)} images")

loaded 202597 images


In [15]:
dataloader=DataLoader(dataset,batch_size=128,shuffle=True)

### Generator Network

In [15]:
import torch.nn as nn 
import torch.optim as optim
import numpy as np

In [18]:
class Generator(nn.Module):
    def __init__(self,img_channels=3,z_dim=100):
        super(Generator,self).__init__()

        # fully connected(dense) Layers

        self.model=nn.Sequential(
            nn.Linear(100,256),
            nn.ReLU(),

            nn.Linear(256,512),
            nn.ReLU(),

            nn.Linear(512,1024),
            nn.ReLU(),

            nn.Linear(1024,64*64*img_channels),
            nn.Tanh() # [-1,1]
        )

    def forward(self,z):
        img=self.model(z)
        img=img.view(img.size(0),3,64,64)
        return img